In [3]:
import pandas as pd 

df = pd.read_csv('data/spam.csv', encoding='latin-1')
df.head()

,message,is_spam
0,"Go until jurong point, crazy.. Available only ...",False
1,Ok lar... Joking wif u oni...,False
2,Free entry in 2 a wkly comp to win FA Cup fina...,True
3,U dun say so early hor... U c already then say...,False
4,"Nah I don't think he goes to usf, he lives aro...",False


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   message  5572 non-null   object
 1   is_spam  5572 non-null   bool  
dtypes: bool(1), object(1)
memory usage: 49.1+ KB


In [6]:
df['is_spam'].value_counts()

is_spam
False    4825
True      747
Name: count, dtype: int64

# SMS DATA GENERATION

In [6]:
import pandas as pd
import json
import os
import random
from datetime import datetime
import ollama

# 1. Configuration
MODEL_NAME = "gpt-oss:20b-cloud" 
GEN_SIZE = 1001
TIMESTAMP = datetime.now().strftime("%d_%m_%Hh") 
FOLDER_NAME = f"sim_data_{TIMESTAMP}_GEN_{GEN_SIZE}"
ERROR_FOLDER = os.path.join(FOLDER_NAME, "broken_pairs")

# 2. Load and Sample Data
df = pd.read_csv('data/spam.csv', encoding='latin-1')
df = df.iloc[:, [0, 1]] 
df.columns = ['label', 'text']
samples = df.sample(GEN_SIZE)

# 3. Create Folders
os.makedirs(FOLDER_NAME, exist_ok=True)
os.makedirs(ERROR_FOLDER, exist_ok=True)

print(f"Starting generation in folder: {FOLDER_NAME}")

# 4. Define the Example Pool (List of dictionaries/strings)
FEW_SHOT_POOL = [
    """Example (Student/University Ham vs. Fake Scholarship Spam):
    {
        "ham": "Dear student, the deadline for your Semester 6 project submission has been extended to 28-02-26. Check the portal for details. Regards, Admin.",
        "spam": "UGC Govt Scholarship of Rs. 45,000 has been APPROVED for your profile! Clik here to verify bank details and claim amount before link expires <URL>"
    }""",
    """Example (Local Utility Ham vs. Authority Impersonation Spam):
    {
        "ham": "Alert: Water supply in your area will be affected tomorrow from 10 AM to 5 PM due to NMMC maintenance work. Please store water.",
        "spam": "MahaPolice: Final Warning. Your vehicle has a pending Lok Adalat fine. Pay Rs 500 immediately to avoid driving license suspension: <URL>"
    }""",
    """Example (E-commerce Ham vs. Greed/Phishing Spam):
    {
        "ham": "Your Flipkart package containing 'Wireless Mouse' is out for delivery today. Share PIN 4021 with the delivery agent.",
        "spam": "Amazon Big Billion Reward! You are the lucky winner of Apple iPhone 15 Pro. Pay only Rs 99 delivery charge to get your phone: <URL>"
    }""",
    """Example (Stock Market/Trading Ham vs. WhatsApp Scam Spam):
    {
        "ham": "Update: Your mandate for SME IPO is successfully created. An amount of Rs 14,500 will be blocked in your linked bank account.",
        "spam": "Earn daily Rs 5000-10000 from home with our VIP trading signals. No experience needed! Join premium WhatsApp group free today: <URL>"
    }""",
    """Example (Telecom Ham vs. Identity Theft/Aadhaar Spam):
    {
        "ham": "Dear Jio user, you have consumed 100% of your daily 2GB data quota. Your internet speed is now reduced to 64kbps.",
        "spam": "Dear citizen, your Aadhaar biometric is currently compromised. Lock it immediately using the official portal to prevent bank fraud <URL>."
    }""",
    """Example (Casual Hinglish Ham vs. Urgent HR/Job Spam):
    {
        "ham": "Are bro kal ka assignment submit kiya kya? Sir ne bola tha marks deduct karenge late submission pe.",
        "spam": "Urgent Requirement: Wipro is hiring remote data entry operators. Salary 30k/month. Send your resume on this link immediately to schedule HR round: <URL>"
    }""",
    """Example (Conversational Ham vs. Logistics Spam):
    {
        "ham": "Bhai, running 10 mins late for the lecture. Proxy laga dena please.",
        "spam": "India Post: Your package is hold at warehouse due to unpaid customs fee of Rs.45. Pay immediately here: <URL> or package will return."
    }""",
    """Example (Banking Ham vs. Greed/Lottery Spam):
    {
        "ham": "Dear Customer, Rs. 15,000.00 is credited to your A/C ...4567 on 24-02-26. Info: NEFT-SALARY. Available Bal: Rs. 42,500.00.",
        "spam": "Congratulation!! U have won Rs 25,00,000 in KBC Jio Lucky Draw. Click link to contact prize department and claim money <URL>"
    }""",
    """Example (Promotional Ham vs. Gov/Utility Spam):
    {
        "ham": "Weekend is here! Get FLAT 50% OFF on your favorite Dominos pizzas today! Use code PIZZA50 on the app. T&C apply.",
        "spam": "E-CHALLAN ALERT: Traffic fine of Rs 1000 is pending for your vehicle. Pay fine before court forwarding via <URL>"
    }"""
]

# 5. Generation Loop
for i, (idx, row) in enumerate(samples.iterrows(), 1):
    original_text = row['text']
    original_label = row['label']
    
    # Dynamically select 3 random examples for this iteration
    selected_examples = random.sample(FEW_SHOT_POOL, 3)
    examples_string = "\n\n".join(selected_examples)

    prompt = f"""
    You are an expert cyber threat intelligence generator.
    I am building a real-time phishing detection system. Your task is to generate a diverse, highly realistic pair of Indian-context SMS messages.

    CRITICAL INSTRUCTIONS:
    1. DIVERSITY IS MANDATORY: Do not repeat the same scenarios. Randomly select from these categories:
       - Spam Vectors: Lottery/KBC wins, work-from-home WhatsApp job offers, fake traffic e-challans, Income Tax refunds, missed courier deliveries, free 5G data upgrades, or urgent KYC/account blocks.
       - Ham Vectors: Standard OTPs, salary credit alerts, IRCTC train ticket confirmations, doctor appointment reminders, genuine Swiggy/Zomato discounts, or casual friend/colleague text messages.
    2. Realism & Language: Use a natural mix of English and 'Hinglish'. Spam should frequently include realistic typos (e.g., 'K Y C', 'Updat', 'Accnt', 'Congratulation'), urgent tones, or greedy hooks. Ham should look like standard automated system alerts or normal human texting.
    3. Link Masking: NEVER generate a real or fake URL. Replace ALL links (malicious or benign) strictly with the exact token: <URL>.
    4. Format: Return ONLY a valid JSON object. Do not include markdown code blocks (like ```json), conversational text, or explanations.
    5. ALPHABET RESTRICTION: Use ONLY the English/Latin alphabet. Even when writing in Hindi or Hinglish, spell the words using English letters (e.g., write "kripya update kare" instead of "कृपया अपडेट करें"). DO NOT under any circumstances use Devanagari or any native Indian Unicode scripts.
    
    --- FEW-SHOT EXAMPLES ---
    {examples_string}
    -------------------------

    Here is a baseline example from my dataset labeled as '{original_label}':
    "{original_text}"

    Using the baseline ONLY for loose inspiration for length, generate a NEW, completely unique JSON pair following the rules and examples above. Ensure the scenario is different from the examples provided.
    """

    try:
        response = ollama.generate(model=MODEL_NAME, prompt=prompt, format="json")
        raw_response = response['response']
        
        data = json.loads(raw_response)
        
        file_path = os.path.join(FOLDER_NAME, f"pair{i}.json")
        # Added UTF-8 and ensure_ascii=False
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
        print(f"✅ Saved: pair{i}.json")

    except Exception as e:
        error_file_path = os.path.join(ERROR_FOLDER, f"broken_pair{i}.txt")
        # Added UTF-8 here too
        with open(error_file_path, 'w', encoding='utf-8') as f:
            f.write(f"Error: {str(e)}\n\n")
            f.write("--- RAW RESPONSE ---\n")
            f.write(raw_response)
        print(f"❌ Broken: pair{i}.json (Logged to broken_pairs folder)")

print(f"\nGeneration complete. Check '{FOLDER_NAME}' for results.")

Starting generation in folder: sim_data_07_04_22h_GEN_1001
✅ Saved: pair1.json


KeyboardInterrupt: 

# DSPY 


In [16]:
import pandas as pd
import json
import os
import random
from datetime import datetime
import dspy
from pydantic import BaseModel, Field

# 1. Configuration
MODEL_NAME = "gpt-oss:20b-cloud" 
GEN_SIZE = 1000
TIMESTAMP = datetime.now().strftime("%d_%m_%Hh") 
FOLDER_NAME = f"sim_data_V2_{TIMESTAMP}_GEN_{GEN_SIZE}"
ERROR_FOLDER = os.path.join(FOLDER_NAME, "broken_pairs")

# Configure DSPy (Using the LiteLLM/Ollama fix)
lm = dspy.LM(
    model=f"ollama/{MODEL_NAME}", 
    api_base="http://localhost:11434", 
    api_key="sk-local", 
    temperature=0.7,
    max_tokens=1000
)
dspy.settings.configure(lm=lm)

# 2. Load Data
df = pd.read_csv('data/spam.csv', encoding='latin-1')
df = df.iloc[:, [0, 1]] 
df.columns = ['label', 'text']
samples = df.sample(GEN_SIZE)

os.makedirs(FOLDER_NAME, exist_ok=True)
os.makedirs(ERROR_FOLDER, exist_ok=True)

# 3. Expanded Vectors
SPAM_VECTORS = [
    "Lottery/KBC win", "WhatsApp remote job offer", "Fake traffic e-challan", 
    "Income Tax refund", "Missed courier delivery", "Free 5G data upgrade", 
    "Urgent bank KYC/account block", "Credit card limit increase scam",
    "Electricity bill disconnection tonight", "Pre-approved personal loan",
    "Fake SBI/HDFC reward points expiry", "PAN card linking final warning",
    "Crypto investment daily returns", "eSIM upgrade fraud"
]

HAM_VECTORS = [
    "Standard bank OTP", "Salary credit alert", "IRCTC train ticket confirmation", 
    "Doctor appointment reminder", "Genuine Swiggy/Zomato discount", 
    "Casual friend checking in", "University assignment deadline", 
    "Flight boarding gate update", "Jio/Airtel recharge successful",
    "Mutual fund SIP deduction", "Society maintenance bill reminder",
    "Broadband engineer visit update", "HR interview scheduling",
    "Library book return notice"
]

# 4. Pydantic Schema
class SMSPair(BaseModel):
    ham: str = Field(description="The highly realistic benign (ham) SMS text.")
    spam: str = Field(description="The highly realistic malicious (spam) SMS text.")

# 5A. Define the Hinglish Signature
class GenerateSMSPairHinglish(dspy.Signature):
    """You are an expert cyber threat intelligence generator. Generate a highly realistic pair of Indian-context SMS messages.
    
    CRITICAL INSTRUCTIONS:
    1. Realism & Language: Use a natural mix of English and 'Hinglish' (Hindi written in English alphabet). Spam should frequently include realistic typos, urgent tones, or greedy hooks.
    2. Link Masking: NEVER generate a real or fake URL. Replace ALL links strictly with <URL>.
    3. ALPHABET RESTRICTION: Use ONLY the English/Latin alphabet (e.g., write "kripya update kare", NOT "कृपया").
    """
    baseline_inspiration = dspy.InputField(desc="A real SMS from the dataset for stylistic inspiration.")
    target_spam_category = dspy.InputField(desc="The specific type of spam scenario to generate.")
    target_ham_category = dspy.InputField(desc="The specific type of ham scenario to generate.")
    generated_pair: SMSPair = dspy.OutputField()

# 5B. Define the English-Only Signature
class GenerateSMSPairEnglish(dspy.Signature):
    """You are an expert cyber threat intelligence generator. Generate a highly realistic pair of Indian-context SMS messages.
    
    CRITICAL INSTRUCTIONS:
    1. Realism & Language: Use STRICTLY standard English. DO NOT use Hindi or Hinglish. Spam should frequently include realistic typos, urgent tones, or greedy hooks. Ham should look like standard automated system alerts or normal English texting.
    2. Link Masking: NEVER generate a real or fake URL. Replace ALL links strictly with <URL>.
    3. ALPHABET RESTRICTION: Use ONLY the English/Latin alphabet.
    """
    baseline_inspiration = dspy.InputField(desc="A real SMS from the dataset for stylistic inspiration.")
    target_spam_category = dspy.InputField(desc="The specific type of spam scenario to generate.")
    target_ham_category = dspy.InputField(desc="The specific type of ham scenario to generate.")
    generated_pair: SMSPair = dspy.OutputField()

# 6. Initialize Both Generators with TypedPredictor
generator_hinglish = dspy.Predict(GenerateSMSPairHinglish)
generator_english = dspy.Predict(GenerateSMSPairEnglish)

print(f"Starting generation in folder: {FOLDER_NAME}")

# 7. Generation Loop
for i, (idx, row) in enumerate(samples.iterrows(), 1):
    original_text = row['text']
    current_spam_cat = random.choice(SPAM_VECTORS)
    current_ham_cat = random.choice(HAM_VECTORS)

    # Alternate between Hinglish and English generators
    if i % 2 == 0:
        active_generator = generator_english
        lang_mode = "English"
    else:
        active_generator = generator_hinglish
        lang_mode = "Hinglish"

    try:
        # Run the DSPy prediction using the active generator
        prediction = active_generator(
            baseline_inspiration=original_text,
            target_spam_category=current_spam_cat,
            target_ham_category=current_ham_cat
        )
        
        validated_pair = prediction.generated_pair
        
        file_path = os.path.join(FOLDER_NAME, f"pair{i}.json")
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(validated_pair.model_dump(), f, indent=4, ensure_ascii=False)
            
        print(f"✅ Saved: pair{i}.json | Mode: {lang_mode} | Ham: [{current_ham_cat}] | Spam: [{current_spam_cat}]")

    except Exception as e:
        print(f"❌ Error in pair{i}: {str(e)}")
        error_file_path = os.path.join(ERROR_FOLDER, f"broken_pair{i}.txt")
        with open(error_file_path, 'w', encoding='utf-8') as f:
            f.write(f"Error: {str(e)}\n\n")
            f.write(f"Prompt History:\n{lm.inspect_history(n=1)}")
            
        print(f"Logged to broken_pairs folder.")

print(f"\nGeneration complete. Check '{FOLDER_NAME}' for results.")

Starting generation in folder: sim_data_V2_07_04_23h_GEN_1000
❌ Error in pair1: 'NoneType' object has no attribute 'model_dump'




[2026-04-07T23:31:10.017597]

System message:

Your input fields are:
1. `baseline_inspiration` (str): A real SMS from the dataset for stylistic inspiration.
2. `target_spam_category` (str): The specific type of spam scenario to generate.
3. `target_ham_category` (str): The specific type of ham scenario to generate.
Your output fields are:
1. `generated_pair` (SMSPair):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## baseline_inspiration ## ]]
{baseline_inspiration}

[[ ## target_spam_category ## ]]
{target_spam_category}

[[ ## target_ham_category ## ]]
{target_ham_category}

Outputs will be a JSON object with the following fields.

{
  "generated_pair": "{generated_pair}        # note: the value you produce must adhere to the JSON schema: {\"type\": \"objec